In [ ]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Clone the original YOLOv9 research repository
!git clone https://github.com/WongKinYiu/yolov9.git
%cd yolov9

# 3. Install the required dependencies
!pip install -r requirements.txt -q
print("✅ YOLOv9 Research Environment Ready!")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Patch the YOLOv9 experimental.py file to allow loading weights in PyTorch 2.6+
!sed -i "s/ckpt = torch.load(attempt_download(w), map_location='cpu')/ckpt = torch.load(attempt_download(w), map_location='cpu', weights_only=False)/g" /content/yolov9/models/experimental.py

print("✅ Successfully patched YOLOv9 for PyTorch 2.6+!")

✅ Successfully patched YOLOv9 for PyTorch 2.6+!


In [ ]:
!pip install "Pillow<10.0.0"

In [ ]:
# Make sure we are in the yolov9 folder
%cd /content/yolov9

print("📊 Running final evaluation on the TEST dataset...")

# FIX: Using the 'dual' script to handle the auxiliary branch outputs
!python val_dual.py \
--data /content/drive/MyDrive/VOC_PCB/yolo_ready/dataset.yaml \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--task test \
--img 640 \
--batch 16 \
--device 0 \
--name pcb_test_eval

print("\n✅ Evaluation complete! The detailed results are saved in /content/yolov9/runs/val/pcb_test_eval")

/content/yolov9
📊 Running final evaluation on the TEST dataset...
val_dual: data=/content/drive/MyDrive/VOC_PCB/yolo_ready/dataset.yaml, weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.7, max_det=300, task=test, device=0, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=pcb_test_eval, exist_ok=False, half=False, dnn=False, min_items=0
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs
test: Scanning /content/drive/.shortcut-targets-by-id/1ERd-i11S5R5TIyYBTyyX0Z6JtO39shm6/VOC_PCB/yolo_ready/labels/test.cache... 2134 images, 0 backgrounds, 0 corrupt: 100% 2134/2134 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP

In [ ]:
import glob
from IPython.display import Image, display

%cd /content/yolov9

print("🎨 Drawing bounding boxes on the test images...")

# FIX: Using the 'dual' detection script
!python detect_dual.py \
--source /content/drive/MyDrive/VOC_PCB/yolo_ready/images/test \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--img 640 \
--conf 0.5 \
--device 0 \
--name final_predictions

print("\n🖼️ Here are your visual predictions:")

# Find where the script saved the images and display the first 3
predicted_images = glob.glob('/content/yolov9/runs/detect/final_predictions/*.jpg')

for img_path in predicted_images[:3]:
    display(Image(filename=img_path, width=600))
    print("-" * 50)

/content/yolov9
🎨 Drawing bounding boxes on the test images...
detect_dual: weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], source=/content/drive/MyDrive/VOC_PCB/yolo_ready/images/test, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.5, iou_thres=0.45, max_det=1000, device=0, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=final_predictions, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs
image 1/2134 /content/drive/.shortcut-targets-by-id/1ERd-i11S5R5TIyYBTyyX0Z6JtO39shm6/VOC_PCB/yolo_ready/images/test/l_light_01_missing_hole_01_1_600.jpg: 640x640 1 missing hole, 79.

In [ ]:
import shutil
import os

# Where YOLO saved the images temporarily
colab_folder = '/content/yolov9/runs/detect/final_predictions'

# Where they will live permanently in your Drive
drive_folder = '/content/drive/MyDrive/VOC_PCB/final_test_predictions'

print("Moving files to Google Drive...")

if os.path.exists(colab_folder):
    # This copies the folder and everything inside it
    shutil.copytree(colab_folder, drive_folder, dirs_exist_ok=True)
    print(f"✅ Success! All bounding box images are safely backed up at: {drive_folder}")
else:
    print("❌ Wait, I can't find the Colab folder. Did the detect_dual.py script finish running?")

Moving files to Google Drive...
✅ Success! All bounding box images are safely backed up at: /content/drive/MyDrive/VOC_PCB/final_test_predictions


In [ ]:
!python detect_dual.py \
--source /content/drive/MyDrive/VOC_PCB/yolo_ready/images/test \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--img 640 \
--conf 0.5 \
--device 0 \
--project /content/drive/MyDrive/VOC_PCB/ \
--name final_test_predictions

detect_dual: weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], source=/content/drive/MyDrive/VOC_PCB/yolo_ready/images/test, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.5, iou_thres=0.45, max_det=1000, device=0, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/VOC_PCB/, name=final_test_predictions, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs
image 1/2134 /content/drive/.shortcut-targets-by-id/1ERd-i11S5R5TIyYBTyyX0Z6JtO39shm6/VOC_PCB/yolo_ready/images/test/l_light_01_missing_hole_01_1_600.jpg: 640x640 1 missing hole, 75.3ms
image 2/2134 /content/drive/.short

In [ ]:
import shutil
import os

# The exact folder where val_dual.py just saved your test results
colab_val_folder = '/content/yolov9/runs/val/pcb_test_eval4'

# Where you want to save them permanently in your Google Drive
drive_val_folder = '/content/drive/MyDrive/VOC_PCB/final_test_eval_graphs'

print("Moving evaluation graphs and metrics to Google Drive...")

if os.path.exists(colab_val_folder):
    shutil.copytree(colab_val_folder, drive_val_folder, dirs_exist_ok=True)
    print(f"✅ Success! All your FYP evaluation graphs are safely backed up at: {drive_val_folder}")
else:
    print("❌ Error: Could not find the Colab folder. Make sure the path matches your output!")

Moving evaluation graphs and metrics to Google Drive...
✅ Success! All your FYP evaluation graphs are safely backed up at: /content/drive/MyDrive/VOC_PCB/final_test_eval_graphs


In [ ]:
# 1. Zip the entire evaluation folder into one file
!zip -r /content/pcb_test_results.zip /content/yolov9/runs/val/pcb_test_eval4

# 2. Copy the single zip file securely into your Google Drive
!cp /content/pcb_test_results.zip /content/drive/MyDrive/VOC_PCB/

print("✅ Successfully zipped and copied! Check your Drive for pcb_test_results.zip")

  adding: content/yolov9/runs/val/pcb_test_eval4/ (stored 0%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch0_pred.jpg (deflated 8%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch1_labels.jpg (deflated 9%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch2_labels.jpg (deflated 8%)
  adding: content/yolov9/runs/val/pcb_test_eval4/R_curve.png (deflated 11%)
  adding: content/yolov9/runs/val/pcb_test_eval4/F1_curve.png (deflated 11%)
  adding: content/yolov9/runs/val/pcb_test_eval4/PR_curve.png (deflated 17%)
  adding: content/yolov9/runs/val/pcb_test_eval4/P_curve.png (deflated 18%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch1_pred.jpg (deflated 8%)
  adding: content/yolov9/runs/val/pcb_test_eval4/confusion_matrix.png (deflated 24%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch0_labels.jpg (deflated 9%)
  adding: content/yolov9/runs/val/pcb_test_eval4/val_batch2_pred.jpg (deflated 8%)
✅ Successfully zipped and copied! Check you